Project done by:
1. James Megadyanto (U2323402C)
2. Mak Chee Cheng (U2210533J)
3. Choo Shun Yu (U2322188E)

Group Name: AI group

Contributions:
Equally contributed

Project Description:
This project focuses on fine-tuning a pretrained NanoGPT model to solve simple algebra and arithmetic problems using reinforcement learning through Direct Preference Optimization (DPO). A dataset of 100,000 paired responses consisting of negative and positive response is used to train preference alignment. The DPO approach enables the model to learn to generate accurate mathematical answers such as addition, subtraction, multiplication, division, and basic algebra with a short reasoning given .Such as 5+7=? The answer is 12 because 5+7 equals 12.
x/4=3,x=? The answer is 12 because 3*4 equals 12.

The 100k data pairs is generated  from this code: [generate_posnegpairs.ipynb](generate_posnegpairs.ipynb)

In [1]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 3060 Laptop GPU


We are using NVIDIA GeForce RTX 3060 GPU to enable parallel processing making the training time faster

In [2]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200 #generate up to this number of token
temperature = 0.8 #control randomness in generation
top_k = 200 #consider the top 200 most likely token 
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

A smaller beta can sharpen the preference but can destabilise and a larger beta soften preference but can cause underfitting so we kept the 0.5 to blalance it out. The base_lr is set at 1e-4 as we want a conservative LR for stable adaptation of small GPT with AdamW and this avoids the exploding updates on short seqeuences. An epoch of 5 and a batch size of 64 gives the policy multiple passes without overfitting since we'll be using 100k pairs. Although our models positional limit is 256, the math prompts we have are short so setting the max length to 64 helps boost throughput, reduces VRAM, and helps concentrate the learning on the answer-relevant tokens as they are near the end.

In [3]:
def compute_logprob(model, input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = model(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss


def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

The functions in the cell is in charge of making the DPO training numerically stable and comparable across examples. The first function compute_logprob scores each sequence by doing next-token preidciton, comptuing cross-entropy per token, masking PAD - 0, and length normalising so we can return an average log-rpob per example which is exactly what DPO needs to comapare the positive vs negative completions in a fair manner. pad_or_truncate keeps the tail (where the answers are at) and right-pads with 0 to a fixed max_length so that they have uniform shapes. get_batchs bascially shuffles the pairs during epoch, slices full size batches, build aligned negative and positive strings with trailing newlines, eondes them, tail-keeps and pads to max_length, moves them to CUDA long tensors, and yields the (neg_tensor, pos_tensor) ready for the DPO loss and all these functions together also makes sure the DPO optimisation is stable and good to go

In [4]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

wte: Embedding(74, 348) -> character-level model. A 74 token vocab means we're not using byte pair encoding and tokens are characters (digits, arithemtic operators, commas, spaces, letters). This is ideal for short arithmetic prompts becasue the model can see each digit or operator explicitly. The 348-dimensional embedding is the model width whcih is big enough to learn patterns like carry or borrow and solve for x questions but still small for fast epochs.
wpe: Embeddings(256, 348) -> max usable context = 256 chats. Our max length = 64 sit well within this so if we every pushed > 256 characters, attention would stop improving because no positions are defined past 256.

In [5]:

from copy import deepcopy 

# load policy model(gpt)
# create reference model from policy model
ref_gpt = deepcopy(gpt)
ref_gpt.eval()
for param in ref_gpt.parameters():
    param.requires_grad = False


We make a frozen copy of the current model and use it for reference during training so at each step we can check if the trainable model likes the good answer more than this frozen copy. This results in stable training since we're always comparing to the same reference instead of an always changing one

In [6]:
# Load data from ./data/pos_neg_pairs.json
import json

with open('./pos_neg_pairs2.json', 'r') as f:
    lines = json.load(f)
    
print('Loaded', len(lines), 'data pairs')
print(lines[:5])

Loaded 100000 data pairs
[{'negative': '9*4=? Sorry, I do not know.', 'positive': '9*4=? The answer is 36 because 9*4 equals 36.'}, {'negative': '47-x=10,x=? Sorry, I do not know.', 'positive': '47-x=10,x=? The answer is 37 because 47-10 equals 37.'}, {'negative': '7+x=39,x=? Sorry, I do not know.', 'positive': '7+x=39,x=? The answer is 32 because 39-7 equals 32.'}, {'negative': '99-57=? Sorry, I do not know.', 'positive': '99-57=? The answer is 42 because 99-57 equals 42.'}, {'negative': '8+19=? Sorry, I do not know.', 'positive': '8+19=? The answer is 27 because 8+19 equals 27.'}]


Our data pairs are algined to the needs of the DPO. For the same input x, we compare the model's sequence log-rpob for y+ vs y- and psuh the policy to prefer y+ relative to a frozen reference (the cell above). Having n = 100k is impotant as more pairs stabilise training, prevent overfitting, and suppresses the model's tendency to ramble and since we are using short strings it fits compfortably under the max length = 64 so bathcing is efficient and comparisons are more fair

In [7]:
# recommend to use the AdamW optimizer 
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR

# AdamW optimizer
optimizer = AdamW(
    gpt.parameters(), 
    lr=base_lr,          # Use Base LR 1e-4
    weight_decay = 1e-2  # Prevents Overfitting
)

# Scheduler to reduce learning rate over time by 30% every 2 epochs
scheduler = StepLR(optimizer, step_size = 2, gamma = 0.7)

Using the AdamW helps decouples the weight decay from the adaptive gradient udpate, gicing us more predictable regularization which is important when the DPO loss is a logistic on log-prob deltas (can spike if steps are too aggresive). Hvaing a modest lr of 1e-4 keeps updates stable for a small GPT on short seqeunces and the weight_decay = 1e-2 counteracts overfitting to the repeated "The asner is x because x" template by shrinking the weights independtly of the gradient direction. The StepLR schedule also reduces the lr by 30% after epochs 2 and 4 so that the nominal lr path is: 1.0e-4 -> 7.0e-5 -> 4.9e-5, which mirrors how preference learning typically behave (larger, exploratory steps early to split the positives from negatives and finer steps later to avoid overshooting once th emodel alredy prefers the right completions)

In [8]:
total_steps = len(lines) // batch_size

for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):

        neg_logprob = compute_logprob(gpt, neg_tensor) - compute_logprob(ref_gpt, neg_tensor)
        pos_logprob = compute_logprob(gpt, pos_tensor) - compute_logprob(ref_gpt, pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()  

    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}, epoch {epoch+1}/{epochs}")


1562it [09:54,  2.63it/s]


Saved checkpoint to ./dpo.pt, epoch 1/5


1562it [14:00,  1.86it/s]


Saved checkpoint to ./dpo.pt, epoch 2/5


1562it [13:49,  1.88it/s]


Saved checkpoint to ./dpo.pt, epoch 3/5


1562it [14:16,  1.82it/s]


Saved checkpoint to ./dpo.pt, epoch 4/5


1562it [14:14,  1.83it/s]


Saved checkpoint to ./dpo.pt, epoch 5/5


The loop peforms DPO training computing sequence log-probs for positive and negative completions for each batch under the trainable policy and the frozen reference. It forms:
Δ+=logπθ​(y+∣x)−logπref​(y+∣x),Δ−=logπθ​(y−∣x)−logπref​(y−∣x), then minimizing −logσ((Δ+−Δ−)/β)−0.1Δ+ with β = 0.5 to push the policy to prefer y+ over y-. With 100k pairs and a batch size of 64, each epoch runs 1562 full iterations. 

In [9]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)

# Test
gpt.eval()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

In [10]:
test_set = [
# Addition
"5+7=?", "3+5=?", "9+9=?", "7+5=?", "9+10=?", "12+8=?", "15+4=?", "6+11=?", "8+7=?", "14+3=?",
# Subtraction
"7-5=?", "13-2=?", "10-2=?", "11-5=?", "20-7=?", "15-9=?", "18-6=?", "12-8=?", "14-4=?", "9-3=?",
# Multiplication
"5*5=?", "7*8=?", "9*3=?", "10*3=?", "8*3=?", "6*7=?", "4*9=?", "12*2=?", "3*11=?", "5*9=?",
# Division
"20/5=?", "18/3=?", "12/4=?", "15/5=?", "9/3=?", "24/6=?", "16/4=?", "10/2=?", "14/7=?", "8/2=?",
#Algebra
"x+5=12,x=?", "7+x=15,x=?", "x-3=4,x=?", "10-x=6,x=?", "x*3=12,x=?", "5*x=20,x=?", "x/4=3,x=?", "16/x=4,x=?", "x+7=14,x=?", "9+x=20,x=?"
]

with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)

        x = torch.tensor([prompt_ids], device=device)
        generated_tokens = gpt.generate(
            x, 
            max_new_tokens = 60, 
            temperature = 0.2, 
            top_k = 2
        )

        output_ids = generated_tokens[0]
        if hasattr(output_ids, 'tolist'):
            tokens_list = output_ids.tolist()
            if isinstance(tokens_list[0], list):
                tokens_list = [token for sublist in tokens_list for token in sublist]
        else:
            tokens_list = output_ids

        # Decode only the generated tokens after the prompt
        completion_tokens = tokens_list[len(prompt_ids):]
        completion_text = decode(completion_tokens).strip()

        # Print result
        print(f"{prompt} {completion_text}")

5+7=? The answer is 12 because 5+7 equals 12.
3+5=? The answer is 8 because 3+5 equals 8.
9+9=? The answer is 18 because 9+9 equals 18.
7+5=? The answer is 12 because 7+5 equals 12.
9+10=? The answer is 19 because 9+10 equals 19.
12+8=? The answer is 20 because 12+8 equals 20.
15+4=? The answer is 19 because 15+4 equals 19.
6+11=? The answer is 17 because 6+11 equals 17.
8+7=? The answer is 15 because 8+7 equals 15.
14+3=? The answer is 17 because 14+3 equals 17.
7-5=? The answer is 1 because 7-5 equals 1.
13-2=? The answer is 11 because 13-2 equals 11.
10-2=? The answer is 7 because 10-2 equals 7.
11-5=? The answer is 6 because 11-5 equals 6.
20-7=? The answer is 13 because 20-7 equals 13.
15-9=? The answer is 16 because 15-9 equals 16.
18-6=? The answer is 10 because 18-6 equals 10.
12-8=? The answer is 6 because 12-8 equals 6.
14-4=? The answer is 10 because 14-4 equals 10.
9-3=? The answer is 6 because 9-3 equals 6.
5*5=? The answer is 25 because 5*5 equals 25.
7*8=? The answer is 

80% success rate

For the testing, we put the model in eval mode and turn off gradient trackign so the results are consistent, faster, and lighter on the memory. We feed the prompts ith the same character tokeniser we trained with so that the digts and math symbols are handled the same way. For the generation, we set the temperature to 0.2 and top_k to 2 (whcih is near-greedy decoding) as this keeps our model focused as we believe that math needs accuracy, not diversity. This allows the model to almost always pick the best next character, but still has just enough flexibility to break ties which are great for numbers. We also allow up to 60 new characters which we think comfortably fits "The answer is n because ..." without cutting anything off. We also only decode what the model adds after the prompt and added a small safety step so decoding doesn't break if the output format changes and we skipped extra smapling tricks to keep the behaviour steady and not over chatty. From our test results we can see that we got 40/50 correct which is an 80% accuracy rate. Addition 10/10, Multiplication 10/10, Division 9/10, but weaker on Subtraction 6/10 and Algebra 5/10. The output shows strong DPO alignment to the target format ("The answer is n becasue ...") and high factuality on airthemtic that doesn't require sumbol re-arrangement. Most errors cluster in subtraction one-offs (10-2 = 7 and 18-6 = 10) and algebra direction mistakes (adding when you should subtract or rhs and lhs movement). WE deliberately used low-entropy decoding as it minimises randomness and reduces off-by-one drift, meaning that the remaining mistakes reflect the model's understanding rather than sampling noise. Overall through this assignment we learnt that using near-greedy decoding is more suitable for keeping the math answers accurate and consistent, minimising randomness while still able to break ties